# Road Anomaly Detection: RoadAnomaly Dataset

**Deep Learning module project**: pixel-level detection of unexpected obstacles (animals, rocks, cones, lost cargo…) in road scenes.

* The dataset **stays on Google Drive** (shared folder `project-deeplearning`) and is reached from VS Code *or* Colab (section 2).
* All reusable code lives in the `roadanomaly/` package, which the **Streamlit testing interface** (`app.py`) also uses.
* The experimental protocol is fixed and identical for every model (same grouped split, resolution, threshold grid), so the results are directly comparable. `DEEPLEARING_FINALE.ipynb` is the first Colab version, kept for reference.

| # | Section | Assignment requirement |
|---|---|---|
| 1 | Environment setup | reproducibility |
| 2 | Data access (Google Drive) | – |
| 3 | Dataset exploration | *Dataset exploration* |
| 4 | Grouped train / validation / test split | *Data preprocessing* |
| 5 | Preprocessing pipeline | *Data preprocessing* |
| 6 | Models: CNN · transfer-learning U-Net · autoencoder | *Model implementation* |
| 7 | Training + **7.4 optimiser / regularisation / hyper-parameter studies** | *Model implementation* |
| 8 | Validation & model selection | *Performance evaluation* |
| 9 | Final test evaluation (precision, recall, F1, IoU, Dice, AUROC, AP) | *Performance evaluation* |
| 10 | Results analysis | *Results analysis* |
| 11 | Technical report generation | *Deliverables* |
| 12 | Streamlit testing interface | UI testing |

### Where the course is used

| Course chapter | Used in this project |
|---|---|
| **1 – Introduction** (neuron, perceptron, MLP, forward / backward propagation, gradient descent, ReLU / sigmoid) | every layer is a learned weighted sum + non-linearity; training = forward pass, loss, back-propagation, gradient step (sections 6–7) |
| **2 – Loss functions** (MSE, binary cross-entropy, chain rule) | BCE (+ Dice) for segmentation, masked MSE for the autoencoder (section 6) |
| **3 – Hyper-parameters, regularisation, optimisation** | early stopping, data augmentation, dropout, L2, batch-norm, SGD / momentum / RMSprop / Adam, random + coarse-to-fine search (**section 7.4**) |
| **4 – CNN** (convolution, padding, stride, pooling, architectures) | all three models are fully convolutional; MobileNetV2 is a CNN architecture pre-trained on ImageNet (section 6) |
| **10 – Sequence models (RNN / LSTM / GRU)** | *not applicable*: a single still image has no temporal order; this would only matter for video (see `DOCUMENTATION.md`, "Why no RNN") |
| *Autoencoder* (only named in the chapter-1 conclusion, no dedicated chapter) | complementary reconstruction-based detector, explained from first principles in section 6.3 and in `DOCUMENTATION.md` |

**How to run:** select a Python kernel where `requirements.txt` is installed (VS Code: *Select Kernel*), then *Run All*.
By default all three models are trained from scratch (about 25 min on a laptop CPU, the dataset is only 60 images); set `TRAINING_PLAN` in section 7 to `"reuse"` to load previous checkpoints instead.

## 1. Environment setup

The first cell locates the project folder (the one that contains `roadanomaly/`) and makes the package importable.
In **Colab**, the project folder must be on your Drive; adapt `COLAB_CODE_DIR` if you put it elsewhere.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    COLAB_CODE_DIR = Path("/content/drive/MyDrive/PROJECT-DEEPLEARNING")  # folder containing roadanomaly/
    os.chdir(COLAB_CODE_DIR)

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "roadanomaly").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project folder (the one containing roadanomaly/).")

os.environ.setdefault("ROADANOMALY_PROJECT_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root    :", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)

In [ ]:
import json
import platform
import shutil

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib.patches import Patch
from PIL import Image

from roadanomaly import config, data, drive_data, inference, metrics, models, report, training

pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 60)
keras.utils.set_random_seed(config.SEED)

gpus = tf.config.list_physical_devices("GPU")
environment = {
    "python": platform.python_version(),
    "system": platform.system(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "devices": [g.name for g in gpus] or ["CPU"],
}
for key, value in environment.items():
    print(f"{key:<11}: {value}")
print("Seed       :", config.SEED)
print("Outputs    :", config.OUTPUT_DIR)
if not gpus:
    print("\nNo GPU: training runs on CPU (fine for this 60-image dataset, slower for the U-Net).")

## 2. Data access: Google Drive

The dataset is **not copied into the project**: it stays in the shared Google Drive folders listed in `config.DRIVE_SOURCES`
(tried in order until one holds the **complete** dataset: 60 images + 60 masks). `drive_data.ensure_dataset()` looks for it in this order:

1. `ROADANOMALY_DATA_DIR` environment variable (explicit path);
2. the local cache `data/` (filled by a previous download, or `/content/roadanomaly_data` in Colab);
3. **Google Drive for Desktop**: the Drive is mounted as a disk (e.g. `G:\My Drive\project-deeplearning`), so files are read straight from Drive;
4. **download from Drive into the cache**: *no setup needed for folders shared as "anyone with the link"*; otherwise the Drive API (Colab sign-in, or an OAuth *Desktop* client in `.secrets/client_secret.json`).

A copy with missing files is never used silently: an incomplete source is skipped and the next one is tried.

Only missing or changed files are downloaded. See `README.md` → *Dataset access* for the one-time setup.

In [ ]:
location = drive_data.ensure_dataset(allow_download=True)
DATA_DIR = location.data_dir

for path in sorted(DATA_DIR.iterdir()):
    print(("[dir]  " if path.is_dir() else "[file] ") + path.name)

## 3. Dataset exploration

### 3.1 Documentation and file structure
Each frame `frames/<name>.webp` has a folder `frames/<name>.labels/` whose `labels_semantic.png` holds the per-pixel label
(**0 = background, 2 = anomaly**). Instance masks and colour visualisations are not used as targets.

In [ ]:
print((DATA_DIR / "readme.md").read_text(encoding="utf-8")[:1800])

In [ ]:
records = data.build_records(DATA_DIR)
print("Image / mask pairs referenced in frame_list.json:", len(records))
print("Example pair:")
print("  image:", records[0]["image_path"].relative_to(DATA_DIR))
print("  mask :", records[0]["mask_path"].relative_to(DATA_DIR))

sample_rgb = data.load_rgb(records[0]["image_path"])
with Image.open(records[0]["mask_path"]) as mask_file:
    sample_labels = np.array(mask_file)
print(f"\nImage: shape={sample_rgb.shape}, dtype={sample_rgb.dtype}, range=[{sample_rgb.min()}, {sample_rgb.max()}]")
print(f"Mask : shape={sample_labels.shape}, dtype={sample_labels.dtype}, labels={np.unique(sample_labels).tolist()}")

### 3.2 Integrity checks and label statistics
For every pair we check: RGB image with 3 channels, 2-D mask, **identical image/mask sizes** (pixel correspondence) and only documented labels.

In [ ]:
integrity = data.check_integrity(records)
if integrity["errors"]:
    raise ValueError(f"Fix these pairs before continuing: {integrity['errors']}")

exploration = pd.DataFrame(integrity["records"]).drop(columns=["image_path", "mask_path"])
by_name = {r["name"]: r for r in integrity["records"]}
families = sorted(exploration["family"].unique())
global_fraction = integrity["global_anomaly_fraction"]

print("Valid pairs          :", integrity["valid_pairs"])
print("Image shapes         :", integrity["image_shapes"])
print("Mask shapes          :", integrity["mask_shapes"])
print("Pixels per label     :", integrity["label_pixels"])
print("Images w/o anomaly   :", integrity["images_without_anomaly"])
print(f"Global anomaly share : {global_fraction:.2%}")
print(f"'Always background' accuracy: {1 - global_fraction:.2%}  -> accuracy is misleading here")

exploration.groupby("family")["anomaly_fraction"].agg(["count", "mean", "median", "min", "max"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].bar(["background", "anomaly"], [100 * (1 - global_fraction), 100 * global_fraction], color=["steelblue", "tomato"])
axes[0].set_ylabel("% of pixels")
axes[0].set_title("Global label distribution")

axes[1].hist(exploration["anomaly_fraction"] * 100, bins=12, color="tomato", edgecolor="black")
axes[1].set_xlabel("anomaly area per image (%)")
axes[1].set_ylabel("number of images")
axes[1].set_title("Anomaly size varies a lot between images")

axes[2].boxplot([exploration.loc[exploration.family == f, "anomaly_fraction"] * 100 for f in families])
axes[2].set_xticks(range(1, len(families) + 1), families)
axes[2].set_ylabel("anomaly area (%)")
axes[2].set_title("Anomaly area per object family")
fig.tight_layout()
plt.show()

for p in (0, 25, 50, 75, 100):
    print(f"percentile {p:>3}: {np.percentile(exploration['anomaly_fraction'], p):.2%}")

### 3.3 Image ↔ ground-truth correspondence
Smallest, median and largest anomaly. The red overlay checks visually that the masks are aligned with the objects.

In [ ]:
def show_annotations(rows, alpha=0.45):
    fig, axes = plt.subplots(len(rows), 3, figsize=(15, 3.6 * len(rows)), squeeze=False)
    for ax_row, row in zip(axes, rows):
        rgb = data.load_rgb(row["image_path"])
        mask = data.load_native_mask(row["mask_path"])
        ax_row[0].imshow(rgb)
        ax_row[0].set_title(f"{row['name']}\nanomaly area {row['anomaly_fraction']:.2%}", fontsize=9)
        ax_row[1].imshow(mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax_row[1].set_title("ground truth (white = anomaly)")
        ax_row[2].imshow(inference.overlay_mask(rgb, mask, alpha=alpha))
        ax_row[2].set_title("overlay")
        for ax in ax_row:
            ax.axis("off")
    fig.tight_layout()
    plt.show()


ordered = sorted(integrity["records"], key=lambda r: r["anomaly_fraction"])
show_annotations([ordered[0], ordered[len(ordered) // 2], ordered[-1]])

In [ ]:
SHOW_FULL_GALLERY = False  # True = all 60 images (slower)

gallery = integrity["records"] if SHOW_FULL_GALLERY else [
    next(r for r in integrity["records"] if r["family"] == f) for f in families
]
n_cols = 3
n_rows = int(np.ceil(len(gallery) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.2 * n_rows), squeeze=False)
for ax, row in zip(axes.ravel(), gallery):
    rgb = data.load_rgb(row["image_path"])
    ax.imshow(inference.overlay_mask(rgb, data.load_native_mask(row["mask_path"])))
    ax.set_title(f"{row['family']}: {row['name'][:45]}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("One example per family (anomaly in red)")
fig.tight_layout()
plt.show()

### 3.4 Near-duplicate scenes
Several images come from the same scene or photo series (lost tyres, cardboard hedgehogs, rock series, Broadmoor cones).
If they were spread across partitions the test score would be inflated → they are **grouped** before splitting (section 4).

In [ ]:
show_annotations([by_name["lost_tires08.webp"], by_name["lost_tires09.webp"]])
show_annotations([by_name["obstacles05_cardboard_czech_hedgehog.webp"], by_name["obstacles06_cardboard_czech_hedgehog.webp"]])

**Exploration findings**

* 60 valid pairs, all 1280 × 720, labels limited to {0, 2}: images and masks correspond pixel-to-pixel.
* Every image contains an anomaly; anomalies cover ≈ 10 % of the pixels overall, from < 1 % (a small bird) to > 40 % (close-up animals) → **strong class imbalance and scale variation**. Pixel accuracy is therefore not informative; we use precision, recall, F1/Dice and IoU.
* Families: animals dominate; cones, obstacles (rocks, hedgehogs), lost tyres and vehicles are rare → some families will appear only once in validation/test.
* The dataset was designed by its authors as a **test benchmark** (Lis et al., 2019). Training on part of it is a pedagogical supervised protocol; results are not directly comparable to the paper's benchmark.

## 4. Grouped train / validation / test split

Scene groups (`config.SCENE_GROUPS`) are kept together; the remaining images form singleton groups.
Groups are shuffled with seed 42 and assigned 70 / 15 / 15 %. The split is **identical to the original Colab experiment**, which allows reusing its checkpoints.

In [ ]:
split = data.grouped_split(records)
train_records, val_records, test_records = (split["records"][s] for s in data.SPLITS)

rows = []
for s in data.SPLITS:
    subset = exploration[exploration.name.isin(split["split_names"][s])]
    rows.append({
        "split": s,
        "images": len(subset),
        "groups": len(split["split_groups"][s]),
        "mean anomaly area": subset.anomaly_fraction.mean(),
        **subset.family.value_counts().to_dict(),
    })
pd.DataFrame(rows).fillna(0).set_index("split")

In [ ]:
# Experiments already available (e.g. the original Colab run copied in the project folder)
experiment_paths = inference.discover_experiments(config.PROJECT_ROOT)


def same_partitions(experiment):
    return bool(experiment.splits) and all(
        sorted(experiment.splits[s]) == split["split_names"][s] for s in data.SPLITS
    )


for path in experiment_paths:
    exp = inference.load_experiment(path)
    print(f"{exp.label()}\n    identical partitions: {same_partitions(exp)}")
if not experiment_paths:
    print("No previous experiment found: all models will be trained in this session.")

split_path = config.OUTPUT_DIR / "partitioning" / f"grouped_splits_seed{config.SEED}.json"
training.write_json(split_path, {
    "seed": config.SEED,
    "method": "random split of declared scene groups",
    "train_group_fraction": config.TRAIN_GROUP_FRACTION,
    "validation_group_fraction": config.VAL_GROUP_FRACTION,
    "scene_groups": config.SCENE_GROUPS,
    "group_by_name": split["group_by_name"],
    "splits": split["split_names"],
})
print("\nSplit saved:", split_path)

## 5. Preprocessing pipeline

* **Resolution 640 × 360** (½ of native): keeps the 16:9 aspect ratio, fits CPU/GPU memory, and (checked below) no annotation disappears.
* Images: bilinear resize, `float32` in [0, 1]. Masks: **nearest-neighbour** resize (keeps labels binary), 0/2 → 0/1.
* Evaluation always happens at **native resolution**: predicted probabilities are upsampled (bilinear) to 1280 × 720 and compared with the original masks.
* Optional augmentation (training only): horizontal flip applied jointly to image and mask, plus small brightness/contrast/saturation jitter.

In [ ]:
H, W = config.IMG_HEIGHT, config.IMG_WIDTH
sample = min(train_records, key=lambda r: by_name[r["name"]]["anomaly_fraction"])
prepared_image, prepared_mask = data.load_image_mask(sample)
native_rgb = data.load_rgb(sample["image_path"])
native_mask = data.load_native_mask(sample["mask_path"])

print("Sample (smallest training anomaly):", sample["name"])
print("Prepared image:", prepared_image.shape, prepared_image.dtype, "range", prepared_image.min(), prepared_image.max())
print("Prepared mask :", prepared_mask.shape, "labels", np.unique(prepared_mask).tolist())
print(f"Anomaly area native {native_mask.mean():.4%} vs prepared {prepared_mask.mean():.4%}")

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, img, title in zip(
    axes.ravel(),
    [native_rgb, native_mask, prepared_image, prepared_mask[..., 0]],
    ["native image", "native mask", f"prepared image {W}x{H}", "prepared mask"],
):
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
smallest = []
for record in records:
    _, mask = data.load_image_mask(record)
    smallest.append((int(mask.sum()), record["name"]))
smallest.sort()
print("Smallest anomalies after resizing (pixels):", smallest[:3])
if smallest[0][0] == 0:
    raise ValueError("Resizing removes an annotation entirely: increase the resolution.")
print("No annotation disappears at", (H, W))

In [ ]:
preview_ds = data.make_dataset(train_records, training=True, augmentation=True)
batch_images, batch_masks = next(iter(preview_ds))

tf.debugging.assert_shapes([(batch_images, ("B", H, W, 3)), (batch_masks, ("B", H, W, 1))])
tf.debugging.assert_greater_equal(batch_images, 0.0)
tf.debugging.assert_less_equal(batch_images, 1.0)
assert set(np.unique(batch_masks.numpy())) <= {0.0, 1.0}
print("Batch:", batch_images.shape, batch_masks.shape, "-> checks passed")

fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, image, mask in zip(axes, batch_images.numpy(), batch_masks.numpy()):
    ax.imshow(inference.overlay_mask((image * 255).astype(np.uint8), mask[..., 0] > 0))
    ax.axis("off")
fig.suptitle("Augmented training batch (flip + photometric jitter), anomaly in red")
plt.show()

## 6. Models

All three models share the same contract: **RGB image (360, 640, 3) in [0, 1] → anomaly score map (360, 640, 1) in [0, 1]**.
This makes thresholding, evaluation and the Streamlit interface identical for every approach.

### 6.1 Baseline: CNN encoder–decoder trained from scratch
Two conv + max-pool stages (feature extraction, ¼ resolution), a bottleneck, two upsampling + conv stages and a 1×1 sigmoid output.
*Why:* the simplest fully convolutional segmentation network; it measures what can be learnt from 39 training images without prior knowledge.

In [ ]:
baseline_preview = models.build_baseline_cnn()
baseline_preview.summary()

### 6.2 Transfer learning: U-Net with a MobileNetV2 encoder (ImageNet)
The encoder is MobileNetV2 pretrained on ImageNet (frozen at first), the decoder upsamples ×2 four times and concatenates **skip connections** from 4 encoder resolutions (U-Net), then a final upsampling + sigmoid. Inputs are padded to 384 px height (multiple of 32) and the padding is cropped from the output.

*Why:* with only 39 training images, generic features learnt on 1.2 M images (edges, textures, object parts: animals are ImageNet classes) are far more reliable than features learnt from scratch; skip connections recover the precise contours lost by downsampling. MobileNetV2 is light (≈ 2.2 M parameters), so it trains on a laptop CPU. Optional fine-tuning of the top encoder layers is available in section 7.

In [ ]:
transfer_preview = models.build_transfer_unet()
transfer_preview.summary(line_length=110)
print("Parameters:", models.count_parameters(transfer_preview))

### 6.3 Complementary approach: convolutional autoencoder (resynthesis idea)
*The autoencoder is not a dedicated chapter of the course (it is only named in the chapter-1 conclusion next to RNN, CNN and GAN), so it is explained here from the concepts that are in the course.*

**What it is.** An autoencoder is a neural network trained to output its own input: an **encoder** (here strided convolutions, chapter 4) compresses the image into a small **latent code**, a **decoder** (transposed convolutions) rebuilds the image from that code. The loss is a regression loss from chapter 2 (**MSE** between input and reconstruction); the optimiser is Adam (chapter 3). Because the code is much smaller than the image (here 45 × 80 × 8 = 28 800 numbers for 691 200 input values, about 24× smaller, and no skip connection), the network cannot copy the input: it must learn what *typical* road scenes look like.

**How it detects anomalies (Lis et al., ICCV 2019, "resynthesis").** An object never seen in the training background is reconstructed badly, so the per-pixel reconstruction error is high exactly there. We:
* train on the **training images only**, with a **masked MSE computed on background pixels only** (the ground-truth anomaly pixels are excluded from the loss, so the network never learns to reproduce an animal or a rock). Note: this uses the *masks* during training, so the model is **weakly supervised**, not fully unsupervised;
* score an image with the per-pixel squared error averaged over RGB, smoothed with a 9 × 9 average pool and divided by the 99th percentile of the background error on the training set, clipped to [0, 1];
* choose the decision threshold on the validation set like for the other models.

*Why include it:* it is a fundamentally different way to detect "the unexpected" (no notion of *which* objects are anomalies), a useful point of comparison with supervised segmentation. **Expected limitation** (and observed, see section 10): a small autoencoder also reconstructs fine textures (vegetation, road markings) poorly, which produces many false alarms; Lis et al. fix this with a semantic segmenter, a GAN and a learned discrepancy network.

In [ ]:
autoencoder_preview = models.build_conv_autoencoder()
autoencoder_preview.summary()

In [ ]:
check_image = batch_images[:1]
for model in (baseline_preview, transfer_preview):
    out = model(check_image, training=False)
    tf.debugging.assert_shapes([(out, (1, H, W, 1))])
    print(f"{model.name:<28} output {tuple(out.shape)}  range [{float(tf.reduce_min(out)):.3f}, {float(tf.reduce_max(out)):.3f}]")
recon = autoencoder_preview(check_image, training=False)
print(f"{autoencoder_preview.name:<28} output {tuple(recon.shape)} (reconstruction)")
print("Structural checks passed.")

## 7. Training

Common protocol: Adam, batch size 4, early stopping on validation loss (best checkpoint restored), artefacts saved in `outputs/runs/<model>/<run_id>/` (config, CSV log, history, summary, learning curves).

`TRAINING_PLAN` controls each model:
* `"reuse"`: take the checkpoint from a previous experiment **with identical partitions** (e.g. the Colab run), with no retraining;
* `"train"`: train from scratch in this session.

**Protocol v2 (anti-overfitting)** versus the original Colab run (BCE, no augmentation, patience 7, frozen encoder): `bce_dice` loss (handles the 90/10 imbalance), flip + photometric augmentation, patience 10, lr 3e-4 for the transfer model, and 15 epochs of encoder fine-tuning (kept only if the validation loss improves). Model selection still uses validation only; the test set is touched once. The original Colab experiment stays in the project for comparison.

In [ ]:
QUICK_RUN = os.environ.get("ROADANOMALY_QUICK_RUN") == "1"  # smoke test: 1 epoch per model
FORCE_TRAIN = os.environ.get("ROADANOMALY_FORCE_TRAIN") == "1"

# v2 protocol (anti-overfitting): everything is retrained. Set a model to "reuse" to keep the original Colab checkpoint.
TRAINING_PLAN = {"baseline": "train", "transfer": "train", "autoencoder": "train"}

HPARAMS = {
    "baseline": {"learning_rate": 1e-3, "loss": "bce_dice", "max_epochs": 80, "patience": 10, "augmentation": True},
    "transfer": {
        "learning_rate": 3e-4, "loss": "bce_dice", "max_epochs": 60, "patience": 10, "augmentation": True,
        "encoder": "MobileNetV2", "pretrained_weights": "imagenet", "encoder_frozen": True,
        "fine_tune_epochs": 15, "fine_tune_layers": 30, "fine_tune_learning_rate": 1e-4,
    },
    "autoencoder": {
        "learning_rate": 1e-3, "loss": "masked_mse_background", "max_epochs": 60, "patience": 10,
        "augmentation": True, "score_percentile": 99.0, "score_smoothing": 9,
    },
}

if FORCE_TRAIN:
    TRAINING_PLAN = {name: "train" for name in TRAINING_PLAN}
if QUICK_RUN:
    for hp in HPARAMS.values():
        hp.update(max_epochs=1, patience=1)
        if hp.get("fine_tune_epochs"):
            hp["fine_tune_epochs"] = 1

RUNS_DIR = config.OUTPUT_DIR / "runs"
checkpoints, run_configs, training_summaries, histories, best_epochs = {}, {}, {}, {}, {}


def run_config_for(name, model, hp):
    return {
        "model_name": model.name, "approach": name, "seed": config.SEED,
        "image_height": H, "image_width": W, "batch_size": config.BATCH_SIZE, **hp,
        "tensorflow_version": tf.__version__, "devices": environment["devices"],
        "splits": split["split_names"], "group_by_name": split["group_by_name"],
    }


def try_reuse(name):
    """Register a checkpoint from a previous experiment with identical partitions."""
    if TRAINING_PLAN[name] != "reuse":
        return False
    for path in experiment_paths:
        exp = inference.load_experiment(path)
        if name in exp.model_names and same_partitions(exp):
            checkpoints[name] = exp.checkpoint(name)
            run_configs[name] = exp.run_configs.get(name, {"approach": name})
            summary_file = exp.directory / f"{name}_training_summary.json"
            if summary_file.is_file():
                training_summaries[name] = json.loads(summary_file.read_text(encoding="utf-8"))
            print(f"{name}: reusing {checkpoints[name]}")
            return True
    print(f"{name}: no reusable checkpoint found -> training")
    return False


def register_training(name, result, run_config):
    checkpoints[name] = result["checkpoint"]
    run_configs[name] = run_config
    training_summaries[name] = result["summary"]
    histories[name] = result["history"]
    best_epochs[name] = result["summary"]["best_epoch"]
    print(f"{name}: {result['summary']}")


print("Plan:", TRAINING_PLAN, "| quick run:", QUICK_RUN)

### 7.1 Baseline CNN

In [ ]:
name = "baseline"
if not try_reuse(name):
    hp = HPARAMS[name]
    keras.utils.set_random_seed(config.SEED)
    model = training.compile_segmentation(models.build_baseline_cnn(), hp["learning_rate"], hp["loss"])
    cfg = run_config_for(name, model, hp)
    result = training.fit_model(
        model,
        data.make_dataset(train_records, training=True, augmentation=hp["augmentation"]),
        data.make_dataset(val_records),
        RUNS_DIR / name / training.new_run_id(), cfg, hp["max_epochs"], hp["patience"],
    )
    register_training(name, result, cfg)

### 7.2 Transfer-learning U-Net (frozen encoder, optional fine-tuning)

In [ ]:
name = "transfer"
if not try_reuse(name):
    hp = HPARAMS[name]
    keras.utils.set_random_seed(config.SEED)
    model = training.compile_segmentation(models.build_transfer_unet(), hp["learning_rate"], hp["loss"])
    cfg = run_config_for(name, model, hp)
    run_dir = RUNS_DIR / name / training.new_run_id()
    train_ds = data.make_dataset(train_records, training=True, augmentation=hp["augmentation"])
    val_ds = data.make_dataset(val_records)
    result = training.fit_model(model, train_ds, val_ds, run_dir, cfg, hp["max_epochs"], hp["patience"])

    if hp["fine_tune_epochs"] > 0:
        # Phase 2: unfreeze the top of the encoder with a 10x smaller learning rate
        model = inference.load_model(result["checkpoint"])
        unfrozen = models.unfreeze_encoder_top(model, hp["fine_tune_layers"])
        print(f"Fine-tuning {unfrozen:,} encoder parameters")
        training.compile_segmentation(model, hp["fine_tune_learning_rate"], hp["loss"])
        fine = training.fit_model(
            model, train_ds, val_ds, run_dir / "fine_tune", {**cfg, "encoder_frozen": False},
            hp["fine_tune_epochs"], hp["patience"], initial_epoch=result["summary"]["epochs_completed"],
        )
        if fine["summary"]["best_validation_loss"] < result["summary"]["best_validation_loss"]:
            fine["history"] = training.merge_histories(result["history"], fine["history"])
            result, cfg = fine, {**cfg, "encoder_frozen": False}
            print("Fine-tuning improved the validation loss -> fine-tuned checkpoint kept")
        else:
            print("Fine-tuning did not improve the validation loss -> frozen-encoder checkpoint kept")
    register_training(name, result, cfg)

### 7.3 Convolutional autoencoder + anomaly scorer
After training, the background-error scale is calibrated on the **training** set and the autoencoder is wrapped into a scorer model that outputs an anomaly map in [0, 1] (saved as a regular `.keras` file).

In [ ]:
name = "autoencoder"
if not try_reuse(name):
    hp = HPARAMS[name]
    keras.utils.set_random_seed(config.SEED)
    autoencoder = training.compile_autoencoder(models.build_conv_autoencoder(), hp["learning_rate"])
    cfg = run_config_for(name, autoencoder, hp)
    run_dir = RUNS_DIR / name / training.new_run_id()
    result = training.fit_model(
        autoencoder,
        data.make_dataset(train_records, training=True, augmentation=hp["augmentation"], target="reconstruction"),
        data.make_dataset(val_records, target="reconstruction"),
        run_dir, cfg, hp["max_epochs"], hp["patience"],
    )
    autoencoder = inference.load_model(result["checkpoint"])
    scale = training.background_error_scale(autoencoder, train_records, hp["score_percentile"], hp["score_smoothing"])
    scorer = models.build_autoencoder_scorer(autoencoder, scale, hp["score_smoothing"])
    scorer_path = run_dir / "anomaly_scorer.keras"
    scorer.save(scorer_path)
    cfg = {**cfg, "model_name": scorer.name, "background_error_scale": scale}
    training.write_json(run_dir / "run_config.json", cfg)
    print(f"Background error scale (p{hp['score_percentile']:.0f}): {scale:.6f}")
    register_training(name, {**result, "checkpoint": scorer_path}, cfg)

In [ ]:
for name, history in histories.items():
    metric_names = [m for m in ("loss", "precision", "recall", "iou") if m in history]
    training.plot_history(
        history, f"{name}: learning curves", best_epochs.get(name),
        path=Path(checkpoints[name]).parent / "learning_curves.png", metrics=metric_names,
    )
    plt.show()
for name in checkpoints:
    if name not in histories:
        print(f"{name}: reused checkpoint (learning curves are in the original run folder)")

### 7.4 Optimisers, regularisation and hyper-parameter search (course chapter 3)

Chapter 3 lists the tools to train a network well; this section applies each of them on the **baseline CNN** (the cheapest model, about 3 min per run on CPU) with a *controlled* protocol: same seed, same loss (BCE + Dice), same augmentation, early stopping (patience 8, at most 40 epochs). Variants are trained on the **train** split and ranked on the **validation** split only (pooled F1 at its best threshold, plus AUROC); the **test set is not used here**.

| Study | Chapter-3 notion | Variants |
|---|---|---|
| Optimisers | SGD, momentum, RMSprop, Adam | lr 1e-2 (SGD, momentum β = 0.9), 1e-3 (RMSprop, Adam) |
| Regularisation | overfitting, dropout, L2 / weight decay, batch-normalisation | none · dropout 0.4 · L2 λ = 1e-4 · BatchNorm · dropout + L2 |
| Hyper-parameter search | random search then **coarse-to-fine** | 5 random trials (lr log-uniform in [1e-4, 1e-2], batch size ∈ {2, 4, 8}), then 3 finer trials around the best |

*Caveat:* one seed and a validation set of 9 images: differences of a few F1 points are within noise. The goal is to show *how* each tool behaves, not to crown a winner.

In [ ]:
from roadanomaly import experiments

RUN_STUDIES = os.environ.get("ROADANOMALY_RUN_STUDIES", "1") == "1" and not QUICK_RUN
# Results are cached in outputs/experiments/<study>/<variant>/result.json: finished variants are not retrained.
if RUN_STUDIES:
    experiments.run_study(experiments.OPTIMIZER_STUDY, "optimizers", train_records, val_records)
    experiments.run_study(experiments.REGULARIZATION_STUDY, "regularization", train_records, val_records)
    experiments.run_search(train_records, val_records)

study_columns = ["name", "optimizer", "learning_rate", "batch_size", "dropout", "l2", "batch_norm",
                 "epochs", "best_epoch", "val_loss", "train_val_gap", "val_f1", "val_iou", "val_auroc", "seconds"]
study_tables = {}
for folder in ("optimizers", "regularization", "search"):
    rows = experiments.load_results(folder)
    study_tables[folder] = pd.DataFrame(rows)[study_columns].set_index("name") if rows else None
study_tables["optimizers"]

In [ ]:
study_tables["regularization"]

In [ ]:
study_tables["search"].sort_values("val_f1", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, (folder, title) in zip(axes, [("optimizers", "Optimisers"), ("regularization", "Regularisation"), ("search", "Random → coarse-to-fine search")]):
    table = study_tables[folder]
    if table is None:
        ax.set_title(f"{title}: not run")
        continue
    table = table.sort_values("val_f1")
    ax.barh(table.index, table["val_f1"], color="tab:blue", label="validation F1")
    ax.scatter(table["val_auroc"], table.index, color="tab:orange", zorder=3, label="validation AUROC")
    ax.set_xlim(0, 1)
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(experiments.EXPERIMENTS_DIR / "summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Validation and model selection

Protocol fixed **before** looking at the test set:
1. predict every validation image, upsample the scores to native resolution;
2. for each model, sweep the thresholds 0.05 … 0.95 and keep the one maximising the **pooled pixel F1** (ties: higher recall, then closest to 0.5);
3. threshold-free metrics are also computed: **AUROC, average precision (AP), FPR at 95 % TPR**, the metrics used by Lis et al.;
4. the primary model is the one with the best validation F1.

In [ ]:
loaded_models = {name: inference.load_model(path) for name, path in checkpoints.items()}
model_summaries = {name: models.count_parameters(m) for name, m in loaded_models.items()}
pd.DataFrame(model_summaries).T

In [ ]:
def predict_split(split_records):
    truths, scores = [], {name: [] for name in loaded_models}
    for record in split_records:
        rgb = data.load_rgb(record["image_path"])
        truths.append(data.load_native_mask(record["mask_path"]))
        for name, model in loaded_models.items():
            scores[name].append(inference.predict_probabilities(model, rgb, (H, W)))
    return truths, scores


val_truth, val_scores = predict_split(val_records)
validation_rows, best_val, val_ranking = {}, {}, {}
for name, scores in val_scores.items():
    pairs = list(zip(val_truth, scores))
    validation_rows[name] = metrics.threshold_sweep(pairs, config.THRESHOLDS)
    best_val[name] = metrics.select_threshold(validation_rows[name])
    val_ranking[name] = metrics.ranking_metrics(pairs)

primary_model = max(best_val, key=lambda n: best_val[n]["f1"])
validation_table = pd.DataFrame({
    name: {**{k: best_val[name][k] for k in ("threshold", "precision", "recall", "f1", "dice", "iou")}, **val_ranking[name]}
    for name in best_val
}).T
print("Primary model selected on validation:", primary_model)
validation_table

In [ ]:
selection_fig, axes = plt.subplots(1, len(validation_rows), figsize=(5.5 * len(validation_rows), 3.8), squeeze=False)
for ax, (name, rows) in zip(axes[0], validation_rows.items()):
    thresholds = [r["threshold"] for r in rows]
    for metric in ("precision", "recall", "f1", "iou"):
        ax.plot(thresholds, [r[metric] for r in rows], marker="o", markersize=3, label=metric)
    ax.axvline(best_val[name]["threshold"], color="gray", linestyle="--", label="selected")
    ax.set_title(name)
    ax.set_xlabel("threshold")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
selection_fig.suptitle("Validation metrics vs. decision threshold")
selection_fig.tight_layout()
plt.show()

In [ ]:
selection_id = training.new_run_id()
selection_dir = config.OUTPUT_DIR / "model_selection" / selection_id
(selection_dir / "models").mkdir(parents=True, exist_ok=False)

selection_payload = {
    "status": "fixed_before_test",
    "selection_dataset": "validation",
    "selection_metric": "pooled_pixel_f1",
    "evaluation_resolution": "native_mask_resolution",
    "probability_resize_method": "bilinear",
    "threshold_comparison": ">",
    "zero_division_value": 0.0,
    "threshold_grid": config.THRESHOLDS.tolist(),
    "primary_model": primary_model,
    "image_height": H,
    "image_width": W,
    "validation_names": sorted(r["name"] for r in val_records),
    "models": {},
}
for name, path in checkpoints.items():
    shutil.copy2(path, selection_dir / "models" / f"{name}.keras")
    selection_payload["models"][name] = {
        "checkpoint_relative_path": f"models/{name}.keras",
        "source_checkpoint": str(path),
        "threshold": best_val[name]["threshold"],
        "validation_metrics": best_val[name],
        "ranking_metrics": val_ranking[name],
    }
    training.write_json(selection_dir / f"{name}_run_config.json", run_configs[name])
    if name in training_summaries:
        training.write_json(selection_dir / f"{name}_training_summary.json", training_summaries[name])
    pd.DataFrame(validation_rows[name]).to_csv(selection_dir / f"{name}_thresholds.csv", index=False)

dataset_summary = {
    "n_images": len(records),
    "native_size": " / ".join(f"{s[1]}x{s[0]}" for s in integrity["image_shapes"]),
    "valid_pairs": integrity["valid_pairs"],
    "errors": len(integrity["errors"]),
    "global_anomaly_fraction": global_fraction,
    "anomaly_fraction_median": float(exploration.anomaly_fraction.median()),
    "anomaly_fraction_min": float(exploration.anomaly_fraction.min()),
    "anomaly_fraction_max": float(exploration.anomaly_fraction.max()),
    "families": exploration.family.value_counts().to_dict(),
}
training.write_json(selection_dir / "dataset_summary.json", dataset_summary)
training.write_json(selection_dir / "model_summaries.json", model_summaries)
training.write_json(selection_dir / "validation_selection.json", selection_payload)
selection_fig.savefig(selection_dir / "validation_threshold_curves.png", dpi=150, bbox_inches="tight")
print("Selection protocol frozen in:", selection_dir)
print("Thresholds:", {n: best_val[n]["threshold"] for n in best_val})

## 9. Final evaluation on the test set

The test images are used **once**, with the thresholds frozen on validation. Metrics are computed from pixel counts pooled over all test images (global), per image (spread) and per object family.

In [ ]:
test_truth, test_scores = predict_split(test_records)
test_names = [r["name"] for r in test_records]
evaluation_dir = selection_dir / "final_evaluation"
predictions_dir = evaluation_dir / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

test_results = {
    "dataset": "test", "primary_model": primary_model, "image_count": len(test_records),
    "evaluation_resolution": "native_mask_resolution", "threshold_comparison": ">",
    "zero_division_value": 0.0, "tensorflow_version": tf.__version__, "models": {},
}
for name, scores in test_scores.items():
    threshold = best_val[name]["threshold"]
    pairs = list(zip(test_truth, scores))
    result = metrics.evaluate(pairs, threshold, names=test_names)
    result["ranking_metrics"] = metrics.ranking_metrics(pairs)

    family_metrics = {}
    for family in families:
        index = [i for i, n in enumerate(test_names) if data.filename_family(n) == family]
        if index:
            counts = np.sum([metrics.confusion_counts(test_truth[i], scores[i] > threshold) for i in index], axis=0)
            family_metrics[family] = {**metrics.scores_from_counts(*counts), "images": len(index)}
    result["family_metrics"] = family_metrics

    for row, image_name, score in zip(result["per_image_metrics"], test_names, scores):
        path = predictions_dir / f"{name}__{Path(image_name).stem}.npz"
        np.savez_compressed(path, probabilities=score.astype(np.float16))
        row["prediction_relative_path"] = str(path.relative_to(selection_dir).as_posix())
    test_results["models"][name] = result
    pd.DataFrame(result["per_image_metrics"]).to_csv(evaluation_dir / f"{name}_per_image_metrics.csv", index=False)

training.write_json(evaluation_dir / "test_results.json", test_results)

test_table = pd.DataFrame({
    name: {"threshold": r["threshold"], **{k: r["global_metrics"][k] for k in ("precision", "recall", "f1", "dice", "iou")}, **r["ranking_metrics"]}
    for name, r in test_results["models"].items()
}).T
print("Test images:", len(test_records), "| primary model (fixed on validation):", primary_model)
test_table

In [ ]:
metric_names = ["precision", "recall", "f1", "iou", "auroc", "average_precision"]
x = np.arange(len(metric_names))
width = 0.8 / len(test_table)
evaluation_fig, ax = plt.subplots(figsize=(12, 4))
for i, (name, row) in enumerate(test_table.iterrows()):
    bars = ax.bar(x + (i - (len(test_table) - 1) / 2) * width, row[metric_names].astype(float), width, label=name)
    ax.bar_label(bars, fmt="%.2f", fontsize=7)
ax.set_xticks(x, metric_names)
ax.set_ylim(0, 1.05)
ax.set_title("Test set: thresholds fixed on validation")
ax.legend()
evaluation_fig.tight_layout()
evaluation_fig.savefig(evaluation_dir / "test_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

curves_fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, scores in test_scores.items():
    curve = metrics.roc_pr_curves(list(zip(test_truth, scores)))
    r = test_results["models"][name]["ranking_metrics"]
    ax_roc.plot(curve["fpr"], curve["tpr"], label=f"{name} (AUROC {r['auroc']:.3f})")
    ax_pr.plot(curve["recall"], curve["precision"], label=f"{name} (AP {r['average_precision']:.3f})")
ax_roc.plot([0, 1], [0, 1], "k:", lw=1)
ax_roc.set(xlabel="false positive rate", ylabel="true positive rate", title="Pixel ROC (test)")
ax_pr.axhline(np.mean(np.concatenate([t.ravel() for t in test_truth])), color="k", ls=":", lw=1, label="chance")
ax_pr.set(xlabel="recall", ylabel="precision", title="Pixel precision-recall (test)")
ax_roc.legend()
ax_pr.legend()
curves_fig.tight_layout()
curves_fig.savefig(evaluation_dir / "roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Results analysis

### 10.1 Validation vs. test
Validation scores are optimistic (they were used to choose checkpoints and thresholds); the gap shows how well the choices generalise.

In [ ]:
analysis_dir = evaluation_dir / "analysis"
analysis_dir.mkdir(exist_ok=True)
rows = []
for name in test_results["models"]:
    for metric in ("precision", "recall", "f1", "iou"):
        v, t = best_val[name][metric], test_results["models"][name]["global_metrics"][metric]
        rows.append({"model": name, "metric": metric, "validation": v, "test": t, "test - validation": t - v})
gap_table = pd.DataFrame(rows).pivot(index="model", columns="metric", values=["validation", "test", "test - validation"])
gap_table.to_csv(analysis_dir / "validation_test_comparison.csv")
gap_table

### 10.2 Per family and per image

In [ ]:
family_table = pd.DataFrame([
    {"model": name, "family": fam, "images": s["images"], "precision": s["precision"], "recall": s["recall"], "f1": s["f1"], "iou": s["iou"]}
    for name, r in test_results["models"].items() for fam, s in r["family_metrics"].items()
]).set_index(["family", "model"]).sort_index()
family_table

In [ ]:
per_image = pd.DataFrame({
    name: {row["name"]: row["f1"] for row in r["per_image_metrics"]} for name, r in test_results["models"].items()
}).sort_values(primary_model)
per_image.to_csv(analysis_dir / "per_image_f1.csv")

ax = per_image.plot.barh(figsize=(11, 0.45 * len(per_image) + 1.5), width=0.8)
ax.set_xlabel("F1 (test, per image)")
ax.set_xlim(0, 1)
ax.set_title("Per-image F1: which scenes are easy / hard for each approach")
plt.tight_layout()
plt.savefig(analysis_dir / "per_image_f1.png", dpi=150, bbox_inches="tight")
plt.show()

print(per_image.describe().loc[["mean", "50%", "min", "max"]].rename(index={"50%": "median"}))

### 10.3 Qualitative comparison
Worst, median and best test images for the primary model. Error maps: **green = true positive, red = false alarm, blue = missed anomaly**.

In [ ]:
def show_comparison(image_name, save=True):
    index = test_names.index(image_name)
    rgb = data.load_rgb(test_records[index]["image_path"])
    truth = test_truth[index]
    names = list(test_scores)
    fig, axes = plt.subplots(2, len(names) + 1, figsize=(4.6 * (len(names) + 1), 6.4))
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("input")
    axes[1, 0].imshow(inference.overlay_mask(rgb, truth))
    axes[1, 0].set_title("ground truth")
    for col, name in enumerate(names, start=1):
        score = test_scores[name][index]
        threshold = best_val[name]["threshold"]
        row = test_results["models"][name]["per_image_metrics"][index]
        axes[0, col].imshow(score, cmap="inferno", vmin=0, vmax=1)
        axes[0, col].set_title(f"{name}: anomaly score")
        axes[1, col].imshow(inference.error_map(truth, score > threshold))
        axes[1, col].set_title(f"t={threshold:.2f}  F1={row['f1']:.2f}  IoU={row['iou']:.2f}")
    for ax in axes.ravel():
        ax.axis("off")
    fig.legend(handles=[Patch(color=np.array(c) / 255, label=l) for c, l in [
        (inference.TP_COLOR, "TP"), (inference.FP_COLOR, "FP (false alarm)"), (inference.FN_COLOR, "FN (missed)")]],
        loc="lower center", ncol=3)
    fig.suptitle(image_name)
    fig.tight_layout(rect=[0, 0.05, 1, 0.96])
    if save:
        fig.savefig(analysis_dir / f"{Path(image_name).stem}_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()


ranked = per_image[primary_model].sort_values().index.tolist()
for image_name in dict.fromkeys([ranked[0], ranked[len(ranked) // 2], ranked[-1]]):
    show_comparison(image_name)

In [ ]:
if "autoencoder" in loaded_models:
    scorer = loaded_models["autoencoder"]
    examples = ranked[:1] + ranked[-1:]
    fig, axes = plt.subplots(len(examples), 4, figsize=(18, 3.3 * len(examples)), squeeze=False)
    for ax_row, image_name in zip(axes, examples):
        index = test_names.index(image_name)
        rgb = data.load_rgb(test_records[index]["image_path"])
        ax_row[0].imshow(rgb)
        ax_row[0].set_title(image_name[:45], fontsize=9)
        ax_row[1].imshow(inference.autoencoder_reconstruction(scorer, rgb, (H, W)))
        ax_row[1].set_title("autoencoder reconstruction")
        ax_row[2].imshow(test_scores["autoencoder"][index], cmap="inferno", vmin=0, vmax=1)
        ax_row[2].set_title("reconstruction error (score)")
        ax_row[3].imshow(test_truth[index], cmap="gray")
        ax_row[3].set_title("ground truth")
        for ax in ax_row:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(analysis_dir / "autoencoder_reconstructions.png", dpi=120, bbox_inches="tight")
    plt.show()

### 10.4 Discussion

*Reference numbers below come from the original Colab experiment `20261030T170429_728027Z` (test set, 12 images). Re-read them against the tables above after your own run.*

**Comparison of approaches**
* **Transfer U-Net ≫ baseline CNN.** Test F1 0.72 / IoU 0.56 vs. 0.47 / 0.31. The pretrained encoder recognises objects (animals, cones) from very few examples, and the skip connections give sharp contours. The baseline, with a small receptive field (two pooling stages, ~46 k parameters) and no prior knowledge, mostly reacts to colour/texture contrast: it needs a very low threshold (0.15) to reach a decent recall, at the price of many false alarms (precision 0.36).
* **Autoencoder.** It needs no anomaly labels at inference and generalises to "anything unusual", but its error map also highlights textures that a small autoencoder reconstructs poorly (vegetation, road markings, sky/horizon), so precision is expected to be the lowest. AUROC / AP show whether its *ranking* of pixels is informative even when its thresholded F1 is low. This is the core limitation that Lis et al. address with a semantic segmenter + GAN resynthesis + a learnt discrepancy network.

**Errors**
* *False alarms*: road markings, vehicles, people and roadside objects that look "salient", plus shadows.
* *Missed anomalies*: small objects far away (pheasant, small rocks), objects with road-like colour, and the cardboard **hedgehogs** (`obstacles07`, F1 0.14 for the transfer model): an unusual shape absent from training once the whole cardboard group was put in test.
* *Contours*: at 640 × 360 the upsampled masks are slightly blurred; thin parts (legs, poles) are lost.

**Validation → test gap.** Transfer F1 drops from 0.88 to 0.72, the baseline is stable. With 9 validation and 12 test images, a single hard scene (the 5 hedgehog images form 42 % of the test set) moves the pooled scores a lot; the gap reflects a *distribution difference* between the small sets more than classic over-fitting.

**Limitations**
* 60 images only; scores and thresholds have high variance (no confidence intervals).
* RoadAnomaly is a test benchmark: training on 39 of its images is not the open-set setting of the paper (where anomalies are never seen during training).
* Pixels of the same image are correlated; pooled pixel metrics favour large objects → per-image metrics are reported too.

**Possible improvements**
* Pre-train / co-train on a large driving dataset (Cityscapes) and paste synthetic anomalies (data synthesis), the standard practice in road anomaly segmentation.
* `loss="bce_dice"` or focal loss for imbalance; stronger augmentation; fine-tune the encoder top (`fine_tune_epochs`).
* Higher input resolution or multi-scale inference for small objects.
* Combine the supervised score with the autoencoder discrepancy (ensemble), or adopt the full resynthesis pipeline.
* Group-aware cross-validation to obtain mean ± std instead of a single split.

## 11. Technical report

The report generator reads every number from the files saved above (`validation_selection.json`, `test_results.json`, run configs, summaries) and copies the figures, so it can be regenerated at any time. Complete the two `[TO COMPLETE]` paragraphs (critical analysis, conclusion) with your own interpretation.

In [ ]:
AUTHORS = "[Group member names]"
report_path = report.write_report(selection_dir / "validation_selection.json", authors=AUTHORS)
print("Report:", report_path)
print("Figures and evidence copied into:", report_path.parent)

## 12. Streamlit testing interface

The UI (`app.py`) loads the experiments found in the project folder (the newest complete one by default, including this run) and lets you:

* pick a **dataset image** (train / validation / test) or **upload your own** road photo;
* run one or several models, adjust the **decision threshold** live, and see the score heatmap, the predicted mask and, when the ground truth exists, the error map with precision / recall / F1 / IoU / Dice;
* browse the dataset and the saved experiment results.

Launch it from a terminal at the project root:

```bash
streamlit run app.py
```

In [ ]:
print("Experiment that the Streamlit app will open by default:")
print(" ", inference.discover_experiments(config.PROJECT_ROOT)[0] if inference.discover_experiments(config.PROJECT_ROOT) else selection_dir)
print("\nCommand:  streamlit run app.py")